# Load Packages

In [ ]:
%load_ext autoreload
%autoreload 2

# Important libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from os.path import join
import joblib
import sys
import torch
sys.path.append("../../")

from src.configs.default_configs import fn_model, fn_pred, fn_pred_perf, device
from src.configs.crc_config import data_name
from src.file_manager.filepath import FilePath
from src.training.misc import get_pos_weight
from src.models.rue.model import AE_Predictor
from src.models.rue.training import train_classifier, train_decoder
from src.training.tuning import tune_model
from src.data_processing.dataloader import get_tabular_dl_dict
from src.training.train import train_model_w_best_param_tabular
from src.models.rue.predicting import get_all_predictions
from src.evaluation.evaluate import get_model_performance_tabular
from src.models.de_mlp.train import train_ensemble_w_best_param_tabular
from src.models.de_mlp.predict import get_all_ensemble_predictions
from seed_file import seed
# seed = 2024

batch_size = 32
eval_batch_size = 128
seed_interval_size, ensemble_size = 100, 5

tuning_seed = 2024

fp = FilePath(data_name=data_name, seed=seed)
fp_scaler_file = join(fp.get_preprocessed_folder(), f"minmax_scaler.pickle")
fp_encoder_file = join(fp.get_preprocessed_folder(), f"encoder.pickle")
fp_split_dict_file = join(fp.get_preprocessed_folder(), f"split_dict.joblib")
fp_split_dict_oversampled_file = join(fp.get_preprocessed_folder(), f"split_dict_oversampled.joblib")

# Load Data

In [ ]:
split_dict_scaled = joblib.load(fp_split_dict_file)
feat_cols_w_pc = ['Age_interview', 'PC1', 'PC2', 'PC3', 'BMI', 'telomere length', 'aHEI2010score', 'aMED', 'DASH', 'SBP', 'DBP', 'Leisure screen time', 'z_pgs000055', 'z_pgs000734', 'Sex (0=Male, 1=Female)', 'alcohol_DailyandWeekly(1)vsMonthlyandNonDrinkers(0)', 'smoke_ex(1)', 'smoke_current(2)', 'Prevalent_diabetes']
target_col = "colorectal cancer"

# Train Ensemble

In [ ]:
pos_weight = get_pos_weight(split_dict_scaled, target_col)
train_param_dict = dict(max_epochs=500, lr=0.001, weight_decay=0.0001, patience=10)
classifier_best_param={'decoder_width': 128, 'encoder_width': 150, 'num_decoder_layers': 3, 'num_encoder_layers': 2}
train_ensemble_w_best_param_tabular(
    ModelClass=AE_Predictor, best_param=classifier_best_param, 
    feature_cols=feat_cols_w_pc, target_col=target_col, 
    train_param_dict=train_param_dict, train_model_func=train_classifier, 
    split_dict=split_dict_scaled, pytorch_split_dict_func=get_tabular_dl_dict,
    seed=seed, seed_interval_size=seed_interval_size, ensemble_size=ensemble_size, data_name=data_name,
    batch_size=batch_size, eval_batch_size=eval_batch_size,
    metric_to_monitor="ce loss", maximise=False, class_weight=pos_weight
)

# Prediction

In [ ]:
split_dict_pytorch = get_tabular_dl_dict(
    **split_dict_scaled, feat_cols=feat_cols_w_pc, target_col=target_col, shuffle_train=False,
    batch_size=batch_size, eval_batch_size=eval_batch_size
)
pred_df_ensemble = get_all_ensemble_predictions(
    **split_dict_pytorch, 
    feature_cols=feat_cols_w_pc, target_col=target_col, seed=seed,
    seed_interval_size=seed_interval_size, ensemble_size=ensemble_size, data_name=data_name,
)
fp_ensemble_predictions_file = join(fp.get_parent_folder(fn_pred), "de.csv")
pred_df_ensemble.to_csv(fp_ensemble_predictions_file)

# Performance Evaluation 

In [ ]:
perf_df = get_model_performance_tabular(
    all_pred_df=pred_df_ensemble, target_col=target_col, label="de")
display(perf_df)
fp_ensemble_perf_file = join(fp.get_parent_folder("perf_evaluation"), "de.csv")
perf_df.to_csv(fp_ensemble_perf_file)